In [1]:
import pandas as pd
import numpy as np

data = pd.read_excel("../data/raw/online_retail_II.xlsx", engine = "openpyxl")

In [2]:
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
data = data[(data["Quantity"] > 0) & (data["Price"] > 0)].copy()
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])
data["Date"] = data["InvoiceDate"].dt.date
data["Date"] = pd.to_datetime(data["Date"])
data["StockCode"] = data["StockCode"].astype(str)
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Date
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12-01
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01


In [4]:
data["Revenue"] = data["Quantity"] * data["Price"]
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Date,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12-01,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,30.0


In [5]:
daily_business = (
    data.groupby(["StockCode", "Date"], as_index = False)
    .agg(
        Quantity = ("Quantity", "sum"),
        Revenue = ("Revenue", "sum"),
        AvgPrice = ("Price", "mean"),
        MedianPrice = ("Price", "median"),
        Transactions = ("Invoice", "nunique")
    )
)

daily_business.head()

,StockCode,Date,Quantity,Revenue,AvgPrice,MedianPrice,Transactions
0,10002,2009-12-01,12,10.20,0.85,0.85,1
1,10002,2009-12-03,7,5.95,0.85,0.85,3
2,10002,2009-12-04,73,62.05,0.85,0.85,4
3,10002,2009-12-06,49,41.65,0.85,0.85,2
4,10002,2009-12-07,2,1.70,0.85,0.85,1


In [6]:
daily_business["WeightedPrice"] = (
    daily_business["Revenue"] / daily_business["Quantity"]
)
daily_business.head()

,StockCode,Date,Quantity,Revenue,AvgPrice,MedianPrice,Transactions,WeightedPrice
0,10002,2009-12-01,12,10.20,0.85,0.85,1,0.85
1,10002,2009-12-03,7,5.95,0.85,0.85,3,0.85
2,10002,2009-12-04,73,62.05,0.85,0.85,4,0.85
3,10002,2009-12-06,49,41.65,0.85,0.85,2,0.85
4,10002,2009-12-07,2,1.70,0.85,0.85,1,0.85


In [7]:
# Top 50 products
forecast_data = pd.read_csv("../data/processed/demand_timeseries.csv")
forecast_data["StockCode"] = forecast_data["StockCode"].astype(str)
forecast_data.head()

,Unnamed: 0,Date,StockCode,Quantity
0,0,2009-12-01,15036,55.0
1,1,2009-12-02,15036,0.0
2,2,2009-12-03,15036,240.0
3,3,2009-12-04,15036,12.0
4,4,2009-12-05,15036,12.0


In [8]:
top_50_products = forecast_data["StockCode"].unique()

daily_business_50 = daily_business[
    daily_business["StockCode"].isin(top_50_products)
].copy()

daily_business_50.head()

,StockCode,Date,Quantity,Revenue,AvgPrice,MedianPrice,Transactions,WeightedPrice
876,15036,2009-12-01,55,35.75,0.65,0.65,2,0.650000
877,15036,2009-12-02,15,19.50,1.30,1.30,2,1.300000
878,15036,2009-12-03,243,119.10,0.89,0.89,2,0.490123
879,15036,2009-12-04,12,7.80,0.65,0.65,1,0.650000
880,15036,2009-12-05,12,7.80,0.65,0.65,1,0.650000


In [9]:
daily_business_50.to_csv(
    "../data/processed/product_daily_business.csv",
    index=False
)

##### Estimate price elasticity

Use a log-log demand model:

log(Quantity) = α + β log(Price) + controls

The coefficient β is the price elasticity.

In [10]:
import statsmodels.api as sm

In [11]:
def estimate_product_elasticity(
        group, min_obs = 30, min_unique_prices = 3
):
    group = group.copy()

    group = group[
        (group["Quantity"] > 0) &
        (group["WeightedPrice"] > 0)
    ]

    if len(group) < min_obs:
        return {
            "Elasticity": np.nan,
            "ElasticityStatus": "insufficient_observations",
            "NumObservations": len(group),
            "NumUniquePrices": group["WeightedPrice"].nunique()
        }
    
    if group["WeightedPrice"].nunique() < min_unique_prices:
        return {
            "Elasticity": np.nan,
            "ElasticityStatus": "insufficient_observations",
            "NumObservations": len(group),
            "NumUniquePrices": group["WeightedPrice"].nunique()
        }
    
    group["log_quantity"] = np.log(group["Quantity"])
    group["log_price"] = np.log(group["WeightedPrice"])

    group["day_of_week"] = group["Date"].dt.dayofweek
    group["month"] = group["Date"].dt.month

    X = group[["log_price", "day_of_week", "month"]]
    X = pd.get_dummies(X, columns = ["day_of_week", "month"], drop_first = True)
    X = sm.add_constant(X)

    X = X.astype(float)

    y = group["log_quantity"]

    model = sm.OLS(y, X).fit(cov_type = "HC3")

    elasticity = model.params["log_price"]

    return {
        "Elasticity": elasticity,
        "ElasticityStatus": "estimated",
        "NumObservations": len(group),
        "NumUniquePrices": group["WeightedPrice"].nunique(),
        "R2": model.rsquared,
        "PValue": model.pvalues["log_price"]  
    }


In [12]:
elasticity_rows = []

for stock_code, group in daily_business_50.groupby("StockCode"):
    result = estimate_product_elasticity(group)
    result["StockCode"] = stock_code
    elasticity_rows.append(result)

elasticity_df = pd.DataFrame(elasticity_rows)
elasticity_df.head()

/home/white/anaconda3/envs/nexusdemand/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2
/home/white/anaconda3/envs/nexusdemand/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2


,Elasticity,ElasticityStatus,NumObservations,NumUniquePrices,R2,PValue,StockCode
0,-3.649100,estimated,207,82,0.554307,8.185697e-20,15036
1,-15.947927,estimated,42,3,0.768796,2.479931e-07,16014
2,-5.445325,estimated,153,29,0.565506,1.477952e-44,17003
3,-1.473346,estimated,288,131,0.185149,1.000000e+00,20724
4,-1.620311,estimated,304,153,0.207960,2.116760e-03,20725


In [13]:
elasticity_df.isnull().sum()

Elasticity          0
ElasticityStatus    0
NumObservations     0
NumUniquePrices     0
R2                  0
PValue              0
StockCode           0
dtype: int64

In [14]:
elasticity_df["ElasticityFinal"] = elasticity_df["Elasticity"]
elasticity_df.head()

,Elasticity,ElasticityStatus,NumObservations,NumUniquePrices,R2,PValue,StockCode,ElasticityFinal
0,-3.649100,estimated,207,82,0.554307,8.185697e-20,15036,-3.649100
1,-15.947927,estimated,42,3,0.768796,2.479931e-07,16014,-15.947927
2,-5.445325,estimated,153,29,0.565506,1.477952e-44,17003,-5.445325
3,-1.473346,estimated,288,131,0.185149,1.000000e+00,20724,-1.473346
4,-1.620311,estimated,304,153,0.207960,2.116760e-03,20725,-1.620311


In [ ]:
fallback_elasticity = -1.0
# Elasticity Final Algo

# If elasticity is missing, use fallback
elasticity_df.loc[
    elasticity_df["ElasticityFinal"].isna(),
    "ElasticityFinal"
] = fallback_elasticity

# If elasticity is positive, flag it and use fallback
elasticity_df["ElasticityFlag"] = "ok"

elasticity_df.loc[
    elasticity_df["Elasticity"] > 0,
    "ElasticityFlag"
] = "positive_elasticity_replaced"

elasticity_df.loc[
    elasticity_df["Elasticity"] > 0,
    "ElasticityFinal"
] = fallback_elasticity

# Clip extreme values
elasticity_df["ElasticityFinal"] = elasticity_df["ElasticityFinal"].clip(
    lower = -5.0,
    upper = -0.05
)

elasticity_df.head()

,Elasticity,ElasticityStatus,NumObservations,NumUniquePrices,R2,PValue,StockCode,ElasticityFinal,ElasticityFlag
0,-3.649100,estimated,207,82,0.554307,8.185697e-20,15036,-3.649100,ok
1,-15.947927,estimated,42,3,0.768796,2.479931e-07,16014,-5.000000,ok
2,-5.445325,estimated,153,29,0.565506,1.477952e-44,17003,-5.000000,ok
3,-1.473346,estimated,288,131,0.185149,1.000000e+00,20724,-1.473346,ok
4,-1.620311,estimated,304,153,0.207960,2.116760e-03,20725,-1.620311,ok


In [16]:
elasticity_df.to_csv(
    "../data/processed/elasticity_by_product.csv",
    index=False
)

So we want to estimated elasticity using a log-log demand model, then added guardrails for low price variation, positive elasticity artifacts, and extreme coefficients.

In [17]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [18]:
from src.optimization.optimizer import suggest_optimal_price

In [19]:
forecast_summary = pd.read_csv("../data/processed/forecast_summary_7d.csv")
elasticity_df = pd.read_csv("../data/processed/elasticity_by_product.csv")
daily_business = pd.read_csv("../data/processed/product_daily_business.csv")

In [20]:
forecast_summary["StockCode"] = forecast_summary["StockCode"].astype(str)
elasticity_df["StockCode"] = elasticity_df["StockCode"].astype(str)
daily_business["StockCode"] = daily_business["StockCode"].astype(str)

In [21]:
daily_business["Date"] = pd.to_datetime(daily_business["Date"])

In [22]:
current_prices = (
    daily_business.sort_values("Date")
    .groupby("StockCode", as_index = False)
    .tail(1)
    [["StockCode", "WeightedPrice"]]
    .rename(columns = {"WeightedPrice": "CurrentPrice"})
)

current_prices.head()

,StockCode,CurrentPrice
1619,21088,1.66
1653,21096,0.85
7859,84212,0.65
8074,84270,0.21
8317,84568,0.43


In [23]:
# Merge

pricing_input = (
    forecast_summary
    .merge(elasticity_df[["StockCode", "ElasticityFinal", "ElasticityFlag"]], on = "StockCode", how = "left")
    .merge(current_prices, on = "StockCode", how = "left")
)

pricing_input.head()

,StockCode,ForecastHorizonDays,DemandP10,DemandP50,DemandP90,ElasticityFinal,ElasticityFlag,CurrentPrice
0,15036,7,0.000000,39.031284,259.284500,-3.649100,ok,1.280000
1,16014,7,0.270505,1.360883,2.452777,-5.000000,ok,0.420000
2,17003,7,0.000000,13.987839,276.808230,-5.000000,ok,0.226923
3,20724,7,0.000000,129.815948,378.037630,-1.473346,ok,1.004375
4,20725,7,0.000000,209.604828,549.372800,-1.620311,ok,2.803529


In [24]:
# We want to simulate the cost / inventory because UCI dataset doesnt provide for it

# Assumption: 40% gross margin, so cost is 60% of current price.
pricing_input["CostPrice"] = pricing_input["CurrentPrice"] * 0.60

In [25]:
# Simulated inventory Scenario
np.random.seed(42)

pricing_input["CurrentInventory"] = (
    pricing_input["DemandP50"] *
    np.random.choice([0.5, 1.0, 2.5], size = len(pricing_input), p = [0.3, 0.4, 0.3])
).round()

pricing_input.head()

,StockCode,ForecastHorizonDays,DemandP10,DemandP50,DemandP90,ElasticityFinal,ElasticityFlag,CurrentPrice,CostPrice,CurrentInventory
0,15036,7,0.000000,39.031284,259.284500,-3.649100,ok,1.280000,0.768000,39.0
1,16014,7,0.270505,1.360883,2.452777,-5.000000,ok,0.420000,0.252000,3.0
2,17003,7,0.000000,13.987839,276.808230,-5.000000,ok,0.226923,0.136154,35.0
3,20724,7,0.000000,129.815948,378.037630,-1.473346,ok,1.004375,0.602625,130.0
4,20725,7,0.000000,209.604828,549.372800,-1.620311,ok,2.803529,1.682118,105.0


In [26]:
# Run optimization:
recommendation_rows = []

for _, row in pricing_input.iterrows():
    result = suggest_optimal_price(
        product_id = row["StockCode"],
        predicted_demand = row["DemandP50"],
        current_inventory = row["CurrentInventory"],
        current_price = row["CurrentPrice"],
        cost_price = row["CostPrice"],
        elasticity = row["ElasticityFinal"],
        min_price_factor = 0.80,
        max_price_factor = 1.20,
        grid_size = 81
    )

    # Remove large optimization table from the final summary.
    result.pop("OptimizationTable")

    recommendation_rows.append(result)

recommendations = pd.DataFrame(recommendation_rows)
recommendations.head()

,ProductID,CurrentPrice,RecommendedPrice,CostPrice,Elasticity,PredictedDemand,CurrentInventory,ExpectedSales,ExpectedRevenue,ExpectedProfit,CurrentProfit,ExpectedProfitUplift,InventoryStatus,RecommendedAction
0,15036,1.280000,1.280000,0.768000,-3.649100,39.031284,39.0,39.000000,49.920000,19.968000,19.968000,0.000000,understock,keep_price
1,16014,0.420000,0.359100,0.252000,-5.000000,1.360883,3.0,2.978450,1.069561,0.318992,0.228628,0.090364,overstock,lower_price
2,17003,0.226923,0.189481,0.136154,-5.000000,13.987839,35.0,34.460235,6.529552,1.837658,1.269665,0.567993,overstock,lower_price
3,20724,1.004375,1.205250,0.602625,-1.473346,129.815948,130.0,99.235419,119.603489,59.801745,52.153557,7.648187,balanced,raise_price
4,20725,2.803529,3.364235,1.682118,-1.620311,209.604828,105.0,105.000000,353.244706,176.622353,117.748235,58.874118,understock,raise_price


In [27]:
recommendations.to_csv(
    "../data/processed/pricing_recommendations.csv",
    index = False
)

In [28]:
summary = (
    recommendations
    .groupby(["InventoryStatus", "RecommendedAction"])
    .agg(
        NumProducts = ("ProductID", "count"),
        AvgProfitUplift = ("ExpectedProfitUplift", "mean"),
        TotalProfitUplift = ("ExpectedProfitUplift", "sum")
    )
    .reset_index()
    .sort_values("TotalProfitUplift", ascending = False)
)

summary

,InventoryStatus,RecommendedAction,NumProducts,AvgProfitUplift,TotalProfitUplift
7,understock,raise_price,24,27.463690,659.128571
3,overstock,lower_price,8,5.858653,46.869228
2,balanced,raise_price,4,6.611699,26.446798
4,overstock,raise_price,3,1.659105,4.977315
0,balanced,keep_price,2,0.000000,0.000000
1,balanced,lower_price,2,0.000000,0.000000
5,understock,keep_price,6,0.000000,0.000000
6,understock,lower_price,1,0.000000,0.000000


In [29]:
# Show the top opportunities

top_opportunities = recommendations.sort_values(
    "ExpectedProfitUplift",
    ascending = False
).head(10)

top_opportunities[
    [
        "ProductID",
        "CurrentPrice",
        "RecommendedPrice",
        "Elasticity",
        "PredictedDemand",
        "CurrentInventory",
        "InventoryStatus",
        "RecommendedAction",
        "ExpectedProfitUplift"
    ]
]

top_opportunities

,ProductID,CurrentPrice,RecommendedPrice,CostPrice,Elasticity,PredictedDemand,CurrentInventory,ExpectedSales,ExpectedRevenue,ExpectedProfit,CurrentProfit,ExpectedProfitUplift,InventoryStatus,RecommendedAction
49,85123A,2.859333,3.288233,1.715600,-5.000000,736.876770,368.0,366.357987,1204.670544,576.146782,420.893867,155.252915,understock,raise_price
29,22386,2.766667,3.320000,1.660000,-3.165862,229.628586,115.0,115.000000,381.800000,190.900000,127.266667,63.633333,understock,raise_price
4,20725,2.803529,3.364235,1.682118,-1.620311,209.604828,105.0,105.000000,353.244706,176.622353,117.748235,58.874118,understock,raise_price
41,84879,1.690000,2.028000,1.014000,-1.624465,677.123108,677.0,503.547556,1021.194444,510.597222,457.652000,52.945222,understock,raise_price
37,84347,4.399091,5.278909,2.639455,-2.487121,111.706955,56.0,56.000000,295.618909,147.809455,98.539636,49.269818,understock,raise_price
42,84946,1.704054,2.044865,1.022432,-2.489572,244.495819,122.0,122.000000,249.473514,124.736757,83.157838,41.578919,understock,raise_price
10,21213,1.280000,1.536000,0.768000,-2.534622,214.134995,107.0,107.000000,164.352000,82.176000,54.784000,27.392000,understock,raise_price
26,22178,1.451600,1.741920,0.870960,-2.246789,173.580719,87.0,87.000000,151.547040,75.773520,50.515680,25.257840,understock,raise_price
13,21731,1.781538,2.137846,1.068923,-3.021926,131.784698,66.0,66.000000,141.097846,70.548923,47.032615,23.516308,understock,raise_price
5,20727,1.650000,1.980000,0.990000,-2.973799,138.855179,69.0,69.000000,136.620000,68.310000,45.540000,22.770000,understock,raise_price
